# EDA — Fonte Externa

## Analise Exploratoria dos Dados Brutos

**Objetivo:** Realizar um overview dos dados disponiveis no bucket de fonte externa (`tech-challenge-fonte-externa-felipe`) antes da execucao do pipeline de ingestao. Esta analise fornece a base para decisoes de transformacao nas camadas Silver e Gold.

**O que este notebook NAO faz:**
- Transformacoes pesadas ou limpeza de dados (responsabilidade da camada Silver)
- Processamento de microdados (`alunos.csv` e inspecionado apenas superficialmente)
- Joins ou cruzamentos entre tabelas (ocorre na camada Gold)

**O que este notebook entrega:**
1. Inventario dos arquivos no bucket
2. Perfil dimensional de cada tabela (linhas, colunas, tipos)
3. Diagnostico de valores nulos e ausentes
4. Cobertura temporal (quais anos cada tabela cobre)
5. Analise do dicionario de dados
6. Validacao do campo `id_municipio` (padrao IBGE) usando o mapeamento UF oficial
7. Estatisticas descritivas das variaveis numericas
8. Conclusoes e implicacoes para o pipeline

---

**Pre-requisitos:**
- AWS CLI configurado com credenciais validas do Learner Lab
- Dependencias: `pandas`, `boto3`

## 1. Setup e Configuracao

In [1]:
import pandas as pd
import boto3
import warnings

warnings.filterwarnings('ignore')

# Configuracao
BUCKET_FONTE = 'tech-challenge-fonte-externa-felipe'

# Whitelist fechada — apenas estes 8 arquivos compoem o escopo
ARQUIVOS_ESPERADOS = [
    'uf.csv',
    'municipio.csv',
    'meta_alfabetizacao_brasil.csv',
    'meta_alfabetizacao_uf.csv',
    'meta_alfabetizacao_municipio.csv',
    'dicionario.csv',
    'alunos.csv',
    'ibge_uf_map.csv',
    'ibge_municipios.csv',
]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 120)

print("Setup concluido")

Setup concluido


## 2. Inventario do Bucket

Verifica quais arquivos existem no bucket e se correspondem a whitelist esperada. Previne erros de leitura por nomes divergentes e documenta o estado da fonte.

In [2]:
s3_client = boto3.client('s3')
resposta = s3_client.list_objects_v2(Bucket=BUCKET_FONTE)
objetos = resposta.get('Contents', [])

arquivos_no_bucket = [obj['Key'] for obj in objetos]

print("=" * 60)
print("  INVENTARIO DO BUCKET")
print("=" * 60)

encontrados = [f for f in ARQUIVOS_ESPERADOS if f in arquivos_no_bucket]
faltando    = [f for f in ARQUIVOS_ESPERADOS if f not in arquivos_no_bucket]
extras      = [f for f in arquivos_no_bucket if f not in ARQUIVOS_ESPERADOS]

print(f"\nArquivos esperados: {len(ARQUIVOS_ESPERADOS)}")
print(f"Encontrados:        {len(encontrados)}")
print(f"Faltando:           {len(faltando)}")
print(f"Extras (fora do escopo): {len(extras)}")

print("\nDetalhamento:")
for obj in objetos:
    nome = obj['Key']
    tamanho_mb = obj['Size'] / (1024 * 1024)
    status = "[OK]" if nome in ARQUIVOS_ESPERADOS else "[EXTRA]"
    print(f"  {status:8s} {nome:45s} {tamanho_mb:8.2f} MB")

if faltando:
    print(f"\nArquivos faltando: {faltando}")
if extras:
    print(f"\nPresentes no bucket mas fora do escopo: {extras}")

  INVENTARIO DO BUCKET

Arquivos esperados: 9
Encontrados:        9
Faltando:           0
Extras (fora do escopo): 5

Detalhamento:
  [OK]     alunos.csv                                      213.66 MB
  [OK]     dicionario.csv                                    0.00 MB
  [OK]     ibge_municipios.csv                               0.12 MB
  [OK]     ibge_uf_map.csv                                   0.00 MB
  [OK]     meta_alfabetizacao_brasil.csv                     0.00 MB
  [OK]     meta_alfabetizacao_municipio.csv                  0.77 MB
  [OK]     meta_alfabetizacao_uf.csv                         0.00 MB
  [OK]     municipio.csv                                     1.36 MB
  [EXTRA]  streaming/staging/93c9fed8-948a-4565-be92-467a7454d14c.json     0.00 MB
  [EXTRA]  streaming/staging/a04b118b-5fee-4c01-ba8d-dbc485bf1cb3.json     0.00 MB
  [EXTRA]  streaming/staging/a89dab95-f647-4f29-871b-8a24aa7ed1a7.json     0.00 MB
  [EXTRA]  streaming/staging/cc257725-496b-41cc-89bb-6c6a5a07cd2d.j

## 3. Carregamento dos Dados

Leitura de todas as tabelas para memoria. Os arquivos sao pequenos (dezenas de MB no total) — nao ha necessidade de amostragem ou leitura em chunks. O `ibge_uf_map.csv` e carregado e tratado da mesma forma que os dados principais.

In [3]:
tabelas = {}

for arquivo in ARQUIVOS_ESPERADOS:
    if arquivo in arquivos_no_bucket:
        nome = arquivo.replace('.csv', '')
        caminho = f's3://{BUCKET_FONTE}/{arquivo}'
        df = pd.read_csv(caminho, low_memory=False)
        tabelas[nome] = df
        print(f"[OK]  {nome:40s} {df.shape[0]:>8,} linhas x {df.shape[1]:>2} colunas")
    else:
        print(f"[ERRO] {arquivo} — nao encontrado no bucket")

print(f"\nTotal de tabelas carregadas: {len(tabelas)}")

[OK]  uf                                            145 linhas x 15 colunas
[OK]  municipio                                  23,995 linhas x 15 colunas
[OK]  meta_alfabetizacao_brasil                       3 linhas x 11 colunas
[OK]  meta_alfabetizacao_uf                          81 linhas x 12 colunas
[OK]  meta_alfabetizacao_municipio               10,704 linhas x 13 colunas
[OK]  dicionario                                     27 linhas x  5 colunas
[OK]  alunos                                   3,867,999 linhas x 12 colunas
[OK]  ibge_uf_map                                    27 linhas x  2 colunas
[OK]  ibge_municipios                             5,571 linhas x  2 colunas

Total de tabelas carregadas: 9


## 4. Perfil Dimensional

Resumo de cada tabela: numero de linhas, colunas e tamanho estimado em memoria. Permite entender rapidamente a escala dos dados e identificar quais tabelas sao as mais volumosas.

In [4]:
resumo_geral = []

for nome, df in tabelas.items():
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    resumo_geral.append({
        'tabela': nome,
        'linhas': df.shape[0],
        'colunas': df.shape[1],
        'memoria_mb': round(mem_mb, 2),
    })

df_resumo = pd.DataFrame(resumo_geral).sort_values('linhas', ascending=False)
print("=" * 60)
print("  PERFIL DIMENSIONAL")
print("=" * 60)
print(df_resumo.to_string(index=False))
print(f"\nTotal de registros: {df_resumo['linhas'].sum():,}")
print(f"Memoria total:      {df_resumo['memoria_mb'].sum():.2f} MB")

  PERFIL DIMENSIONAL
                      tabela  linhas  colunas  memoria_mb
                      alunos 3867999       12      354.13
                   municipio   23995       15        2.75
meta_alfabetizacao_municipio   10704       13        1.15
             ibge_municipios    5571        2        0.15
                          uf     145       15        0.02
       meta_alfabetizacao_uf      81       12        0.01
                  dicionario      27        5        0.00
                 ibge_uf_map      27        2        0.00
   meta_alfabetizacao_brasil       3       11        0.00

Total de registros: 3,908,552
Memoria total:      358.21 MB


## 5. Schema e Tipos de Dados

Para cada tabela, inspecionamos os tipos inferidos pelo pandas e um valor de exemplo.

In [5]:
for nome, df in tabelas.items():
    print("=" * 60)
    print(f"  {nome.upper()}")
    print("=" * 60)
    
    schema = pd.DataFrame({
        'coluna': df.columns,
        'dtype': df.dtypes.values,
        'exemplo': [df[col].dropna().iloc[0] if df[col].notna().any() else 'N/A' for col in df.columns],
    })
    print(schema.to_string(index=False))
    print()

  UF
                 coluna   dtype exemplo
                    ano   int64    2023
               sigla_uf     str      AM
                  serie   int64       2
                   rede   int64       3
     taxa_alfabetizacao float64   49.20
        media_portugues float64  733.66
proporcao_aluno_nivel_0 float64    0.00
proporcao_aluno_nivel_1 float64    2.73
proporcao_aluno_nivel_2 float64    8.59
proporcao_aluno_nivel_3 float64    1.52
proporcao_aluno_nivel_4 float64   13.47
proporcao_aluno_nivel_5 float64   40.41
proporcao_aluno_nivel_6 float64   28.72
proporcao_aluno_nivel_7 float64    3.65
proporcao_aluno_nivel_8 float64    0.92

  MUNICIPIO
                 coluna   dtype    exemplo
                    ano   int64    2023.00
           id_municipio   int64 1100031.00
                  serie   int64       2.00
                   rede   int64       3.00
     taxa_alfabetizacao float64      69.10
        media_portugues float64     767.88
proporcao_aluno_nivel_0 float64       5.8

## 6. Diagnostico de Valores Nulos

Analise sistematica de nulos por tabela e por coluna.

In [6]:
print("=" * 60)
print("  DIAGNOSTICO DE VALORES NULOS")
print("=" * 60)

for nome, df in tabelas.items():
    total = len(df)
    nulos = df.isnull().sum()
    tem_nulos = nulos[nulos > 0]
    
    if len(tem_nulos) == 0:
        print(f"\n[OK]  {nome}: sem valores nulos")
    else:
        print(f"\n[!]   {nome}:")
        for col, qtd in tem_nulos.items():
            pct = qtd / total * 100
            print(f"      {col:35s} {qtd:>8,} nulos ({pct:5.1f}%)")

  DIAGNOSTICO DE VALORES NULOS

[!]   uf:
      proporcao_aluno_nivel_0                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_1                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_2                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_3                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_4                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_5                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_6                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_7                   70 nulos ( 48.3%)
      proporcao_aluno_nivel_8                   70 nulos ( 48.3%)

[!]   municipio:
      proporcao_aluno_nivel_0               11,547 nulos ( 48.1%)
      proporcao_aluno_nivel_1               11,547 nulos ( 48.1%)
      proporcao_aluno_nivel_2               11,547 nulos ( 48.1%)
      proporcao_aluno_nivel_3               11,547 nulos ( 48.1%)
      proporcao_aluno_nivel_4               11,547 nulos ( 48.1%)
      proporcao_

## 7. Cobertura Temporal

Quais anos cada tabela cobre.

In [7]:
print("=" * 60)
print("  COBERTURA TEMPORAL")
print("=" * 60)

for nome, df in tabelas.items():
    if 'ano' in df.columns:
        anos = sorted(df['ano'].dropna().unique().astype(int))
        intervalo = f"{min(anos)}-{max(anos)}"
        print(f"\n{nome}:")
        print(f"  Anos: {anos}")
        print(f"  Intervalo: {intervalo} ({len(anos)} anos)")
        
        dist = df['ano'].value_counts().sort_index()
        for ano_val, qtd in dist.items():
            print(f"    {int(ano_val)}: {qtd:>8,} registros")
    else:
        print(f"\n{nome}: tabela sem coluna 'ano' (dimensao estatica)")

  COBERTURA TEMPORAL

uf:
  Anos: [np.int64(2023), np.int64(2024)]
  Intervalo: 2023-2024 (2 anos)
    2023:       70 registros
    2024:       75 registros

municipio:
  Anos: [np.int64(2023), np.int64(2024)]
  Intervalo: 2023-2024 (2 anos)
    2023:   11,547 registros
    2024:   12,448 registros

meta_alfabetizacao_brasil:
  Anos: [np.int64(2023), np.int64(2024), np.int64(2025)]
  Intervalo: 2023-2025 (3 anos)
    2023:        1 registros
    2024:        1 registros
    2025:        1 registros

meta_alfabetizacao_uf:
  Anos: [np.int64(2023), np.int64(2024), np.int64(2025)]
  Intervalo: 2023-2025 (3 anos)
    2023:       27 registros
    2024:       27 registros
    2025:       27 registros

meta_alfabetizacao_municipio:
  Anos: [np.int64(2023), np.int64(2024)]
  Intervalo: 2023-2024 (2 anos)
    2023:    5,352 registros
    2024:    5,352 registros

dicionario: tabela sem coluna 'ano' (dimensao estatica)

alunos:
  Anos: [np.int64(2023), np.int64(2024)]
  Intervalo: 2023-2024 (2 a

## 8. Analise do Dicionario de Dados

O `dicionario.csv` concentra a traducao de codigos de multiplas tabelas, identificadas pela combinacao `id_tabela` + `nome_coluna`.

In [8]:
df_dicionario = tabelas.get('dicionario')
if df_dicionario is not None:
    print("=" * 60)
    print("  MAPEAMENTO DO DICIONARIO")
    print("=" * 60)
    
    combinacoes = (df_dicionario
                   .groupby(['id_tabela', 'nome_coluna'])
                   .agg(
                       qtd_chaves=('chave', 'nunique'),
                       exemplo_chaves=('chave', lambda x: ', '.join(x.astype(str).unique()[:5])),
                       cobertura=('cobertura_temporal', 'first')
                   )
                   .reset_index()
                   .sort_values(['id_tabela', 'nome_coluna']))
    
    print(combinacoes.to_string(index=False))
    
    # Detalhe das traducoes mais relevantes
for coluna in ['rede', 'serie']:
    subset = df_dicionario[df_dicionario['nome_coluna'] == coluna]
    if len(subset) > 0:
        print(f"\nDecodificacao de '{coluna}':")
        detalhe = subset[['id_tabela', 'chave', 'valor']].drop_duplicates()
        print(detalhe.to_string(index=False))

  MAPEAMENTO DO DICIONARIO
id_tabela           nome_coluna  qtd_chaves exemplo_chaves cobertura
   alunos          alfabetizado           2           0, 1       (1)
   alunos preenchimento_caderno           2           0, 1       (1)
   alunos              presenca           2           0, 1       (1)
   alunos                  rede           4     1, 2, 3, 4       (1)
   alunos                 serie           1              2       (1)
municipio                  rede           7  0, 1, 2, 3, 4       (1)
municipio                 serie           1              2       (1)
       uf                  rede           7  0, 1, 2, 3, 4       (1)
       uf                 serie           1              2       (1)

Decodificacao de 'rede':
id_tabela  chave                                          valor
       uf      0 Total (Federal, Estadual, Municipal e Privada)
municipio      0 Total (Federal, Estadual, Municipal e Privada)
       uf      1                                        Federal
m

## 9. Validacao do Campo `id_municipio`

O campo `id_municipio` segue o padrao IBGE de 7 digitos, onde os **2 primeiros digitos** identificam a UF. O mapeamento de codigos IBGE para siglas de UF e mantido no arquivo `ibge_uf_map.csv` no bucket de fonte externa.

Validacoes:
1. Todos os valores tem exatamente 7 digitos?
2. Os prefixos de 2 digitos correspondem a UFs validas (conforme o CSV)?
3. Distribuicao de municipios por UF

In [9]:
# Extrai o mapeamento a partir da tabela já carregada em memória
df_ibge_uf = tabelas.get('ibge_uf_map')

if df_ibge_uf is not None:
    IBGE_UF_MAP = dict(zip(df_ibge_uf['ibge_code'].astype(int), df_ibge_uf['uf'].astype(str)))
    print(f"Mapeamento UF carregado: {len(IBGE_UF_MAP)} UFs")
else:
    print("[ERRO] Tabela ibge_uf_map nao carregada.")
    IBGE_UF_MAP = {}

for nome in ['municipio', 'meta_alfabetizacao_municipio']:
    df = tabelas.get(nome)
    if df is None or 'id_municipio' not in df.columns:
        continue
    
    print("=" * 60)
    print(f"  VALIDACAO id_municipio - {nome}")
    print("=" * 60)
    
    ids = df['id_municipio'].dropna().astype(str)
    
    # Teste 1: comprimento
    tamanhos = ids.str.len().value_counts().sort_index()
    print(f"\nDistribuicao de comprimento:")
    for tam, qtd in tamanhos.items():
        status = "[OK]" if tam == 7 else "[ERRO]"
        print(f"  {status} {tam} digitos: {qtd:,} registros")
    
    # Teste 2: prefixos validos
    prefixos = ids.str[:2].astype(int)
    prefixos_unicos = sorted(prefixos.unique())
    invalidos = [p for p in prefixos_unicos if p not in IBGE_UF_MAP]
    
    if invalidos:
        print(f"\n[ERRO] Prefixos invalidos encontrados: {invalidos}")
    else:
        print(f"\n[OK]   Todos os {len(prefixos_unicos)} prefixos sao UFs validas")
    
    # Teste 3: distribuicao por UF
    print(f"\nMunicipios distintos por UF:")
    municipios_por_uf = (df[['id_municipio']].drop_duplicates()
                         .assign(uf=lambda x: x['id_municipio'].astype(str).str[:2].astype(int).map(IBGE_UF_MAP))
                         .groupby('uf').size().sort_values(ascending=False))
    
    for uf, qtd in municipios_por_uf.items():
        print(f"  {uf}: {qtd:>5} municipios")
    
    print(f"  {'-' * 25}")
    print(f"  Total: {municipios_por_uf.sum():>4} municipios distintos")
    print()

Mapeamento UF carregado: 27 UFs
  VALIDACAO id_municipio - municipio

Distribuicao de comprimento:
  [OK] 7 digitos: 23,995 registros

[OK]   Todos os 26 prefixos sao UFs validas

Municipios distintos por UF:
  MG:   853 municipios
  SP:   645 municipios
  RS:   493 municipios
  BA:   417 municipios
  PR:   399 municipios
  SC:   295 municipios
  GO:   245 municipios
  PI:   224 municipios
  PB:   223 municipios
  MA:   217 municipios
  PE:   185 municipios
  CE:   184 municipios
  RN:   167 municipios
  PA:   144 municipios
  MT:   141 municipios
  TO:   139 municipios
  AL:   102 municipios
  RJ:    92 municipios
  MS:    79 municipios
  ES:    78 municipios
  SE:    75 municipios
  AM:    62 municipios
  RO:    52 municipios
  AC:    22 municipios
  AP:    16 municipios
  DF:     1 municipios
  -------------------------
  Total: 5550 municipios distintos

  VALIDACAO id_municipio - meta_alfabetizacao_municipio

Distribuicao de comprimento:
  [OK] 7 digitos: 10,704 registros

[OK]   

## 10. Estatisticas Descritivas das Variaveis Numericas

In [10]:
tabelas_numericas = ['municipio', 'uf', 'meta_alfabetizacao_brasil',
                     'meta_alfabetizacao_uf', 'meta_alfabetizacao_municipio']

colunas_interesse = ['taxa_alfabetizacao', 'media_portugues',
                     'meta_alfabetizacao_2024', 'meta_alfabetizacao_2030',
                     'percentual_participacao']

for nome in tabelas_numericas:
    df = tabelas.get(nome)
    if df is None:
        continue
    
    cols = [c for c in colunas_interesse if c in df.columns]
    if not cols:
        continue
    
    print("=" * 60)
    print(f"  {nome.upper()} - Estatisticas Descritivas")
    print("=" * 60)
    print(df[cols].describe().round(2).to_string())
    print()

  MUNICIPIO - Estatisticas Descritivas
       taxa_alfabetizacao  media_portugues
count            23995.00         23995.00
mean                61.44           751.38
std                 19.74            23.23
min                  2.12           673.30
25%                 47.17           736.04
50%                 62.23           750.65
75%                 76.55           764.30
max                100.00           868.46

  UF - Estatisticas Descritivas
       taxa_alfabetizacao  media_portugues
count              145.00           145.00
mean                56.37           745.34
std                 13.19            15.79
min                 30.57           712.56
25%                 47.45           734.25
50%                 55.87           744.82
75%                 63.55           753.30
max                 86.21           797.34

  META_ALFABETIZACAO_BRASIL - Estatisticas Descritivas
       taxa_alfabetizacao  meta_alfabetizacao_2024  meta_alfabetizacao_2030  percentual_participac

## 11. Distribuicao de Variaveis Categoricas

In [11]:
colunas_categoricas = ['rede', 'serie']

for nome, df in tabelas.items():
    cols_presentes = [c for c in colunas_categoricas if c in df.columns]
    if not cols_presentes:
        continue
    
    print("=" * 60)
    print(f"  {nome.upper()} - Variaveis Categoricas")
    print("=" * 60)
    
    for col in cols_presentes:
        valores = df[col].value_counts(dropna=False).head(10)
        print(f"\n  {col} ({df[col].nunique()} valores unicos):")
        for val, qtd in valores.items():
            pct = qtd / len(df) * 100
            label = str(val) if pd.notna(val) else "(NULO)"
            print(f"    {label:20s} {qtd:>8,} ({pct:5.1f}%)")
    print()

  UF - Variaveis Categoricas

  rede (4 valores unicos):
    3                          49 ( 33.8%)
    5                          49 ( 33.8%)
    2                          46 ( 31.7%)
    0                           1 (  0.7%)

  serie (1 valores unicos):
    2                         145 (100.0%)

  MUNICIPIO - Variaveis Categoricas

  rede (4 valores unicos):
    3                      10,896 ( 45.4%)
    5                      10,466 ( 43.6%)
    2                       2,235 (  9.3%)
    0                         398 (  1.7%)

  serie (1 valores unicos):
    2                      23,995 (100.0%)

  META_ALFABETIZACAO_BRASIL - Variaveis Categoricas

  rede (1 valores unicos):
    Pública                     3 (100.0%)

  META_ALFABETIZACAO_UF - Variaveis Categoricas

  rede (1 valores unicos):
    Pública                    81 (100.0%)

  META_ALFABETIZACAO_MUNICIPIO - Variaveis Categoricas

  rede (1 valores unicos):
    Municipal              10,704 (100.0%)

  ALUNOS - Variave

## 12. Consistencia entre Tabelas

In [12]:
print("=" * 60)
print("  CONSISTENCIA ENTRE TABELAS")
print("=" * 60)

# Comparar anos entre tabelas
print("\nAnos por tabela:")
for nome, df in tabelas.items():
    if 'ano' in df.columns:
        anos = sorted(df['ano'].dropna().unique().astype(int))
        print(f"  {nome:40s} {anos}")

# Verificar integridade referencial de id_municipio
df_mun = tabelas.get('municipio')
df_meta_mun = tabelas.get('meta_alfabetizacao_municipio')

if df_mun is not None and df_meta_mun is not None:
    ids_municipio = set(df_mun['id_municipio'].dropna().unique())
    ids_meta = set(df_meta_mun['id_municipio'].dropna().unique())
    
    so_municipio = ids_municipio - ids_meta
    so_meta = ids_meta - ids_municipio
    ambos = ids_municipio & ids_meta
    
    print(f"\nIntegridade referencial (id_municipio):")
    print(f"  Em ambas as tabelas:  {len(ambos):>6,}")
    print(f"  So em 'municipio':    {len(so_municipio):>6,}")
    print(f"  So em 'meta_mun':     {len(so_meta):>6,}")
    
    if so_municipio:
        print(f"  [!] {len(so_municipio)} municipios em 'municipio' sem correspondencia em metas")
    if so_meta:
        print(f"  [!] {len(so_meta)} municipios em metas sem correspondencia em 'municipio'")

  CONSISTENCIA ENTRE TABELAS

Anos por tabela:
  uf                                       [np.int64(2023), np.int64(2024)]
  municipio                                [np.int64(2023), np.int64(2024)]
  meta_alfabetizacao_brasil                [np.int64(2023), np.int64(2024), np.int64(2025)]
  meta_alfabetizacao_uf                    [np.int64(2023), np.int64(2024), np.int64(2025)]
  meta_alfabetizacao_municipio             [np.int64(2023), np.int64(2024)]
  alunos                                   [np.int64(2023), np.int64(2024)]

Integridade referencial (id_municipio):
  Em ambas as tabelas:   5,352
  So em 'municipio':       198
  So em 'meta_mun':          0
  [!] 198 municipios em 'municipio' sem correspondencia em metas


## 13. Inspecao Rapida — Tabela `alunos`

In [13]:
df_alunos = tabelas.get('alunos')

if df_alunos is not None:
    print("=" * 60)
    print("  INSPECAO RAPIDA - ALUNOS (microdados)")
    print("=" * 60)
    
    print(f"\nDimensao: {df_alunos.shape[0]:,} linhas x {df_alunos.shape[1]} colunas")
    mem = df_alunos.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"Memoria:  {mem:.1f} MB")
    
    print(f"\nColunas: {list(df_alunos.columns)}")
else:
    print("[ERRO] Tabela 'alunos' nao encontrada")

  INSPECAO RAPIDA - ALUNOS (microdados)

Dimensao: 3,867,999 linhas x 12 colunas
Memoria:  354.1 MB

Colunas: ['ano', 'id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado', 'proficiencia', 'peso_aluno']


## 14. Verificação de Rede Federal e Privada

Confirma se as categorias `Federal` (código 1) e `Privada` (código 4), previstas no dicionário, de fato ocorrem nos dados reais — tanto nas tabelas com código numérico quanto nas com rede já em texto.

In [14]:
CODIGO_FEDERAL = 1
CODIGO_PRIVADA = 4
TEXTO_PRIVADA = "Privada"

print("=" * 60)
print("  TABELAS COM REDE EM CODIGO NUMERICO")
print("=" * 60)
for nome_tabela in ['municipio', 'uf', 'alunos']:
    df = tabelas[nome_tabela]
    contagem = df['rede'].value_counts().sort_index()
    print(f"\n{nome_tabela}:")
    print(contagem)
    print(f"Federal (1) presente: {CODIGO_FEDERAL in contagem.index}")
    print(f"Privada (4) presente: {CODIGO_PRIVADA in contagem.index}" +
          (f" ({contagem[CODIGO_PRIVADA]} linhas)" if CODIGO_PRIVADA in contagem.index else ""))

print("\n" + "=" * 60)
print("  TABELAS COM REDE EM TEXTO")
print("=" * 60)
for nome_tabela in ['meta_alfabetizacao_brasil', 'meta_alfabetizacao_uf', 'meta_alfabetizacao_municipio']:
    df = tabelas[nome_tabela]
    contagem = df['rede'].value_counts()
    print(f"\n{nome_tabela}:")
    print(contagem)
    print(f"Privada presente: {TEXTO_PRIVADA in contagem.index}")


  TABELAS COM REDE EM CODIGO NUMERICO

municipio:
rede
0      398
2     2235
3    10896
5    10466
Name: count, dtype: int64
Federal (1) presente: False
Privada (4) presente: False

uf:
rede
0     1
2    46
3    49
5    49
Name: count, dtype: int64
Federal (1) presente: False
Privada (4) presente: False

alunos:
rede
2     435398
3    3432576
4         25
Name: count, dtype: int64
Federal (1) presente: False
Privada (4) presente: True (25 linhas)

  TABELAS COM REDE EM TEXTO

meta_alfabetizacao_brasil:
rede
Pública    3
Name: count, dtype: int64
Privada presente: False

meta_alfabetizacao_uf:
rede
Pública    81
Name: count, dtype: int64
Privada presente: False

meta_alfabetizacao_municipio:
rede
Municipal    10704
Name: count, dtype: int64
Privada presente: False


## 15. Verificação da Coluna `nivel_alfabetizacao`

Coluna presente apenas em `meta_alfabetizacao_municipio` (não existe em UF nem Brasil). Verifica se o dicionário decodifica essa escala.

In [15]:
print(df_dicionario[
    (df_dicionario['id_tabela'] == 'meta_alfabetizacao_municipio') &
    (df_dicionario['nome_coluna'] == 'nivel_alfabetizacao')
])

print("\npercentual_participacao presente nas 3 tabelas de meta:")
for nome_tabela in ['meta_alfabetizacao_brasil', 'meta_alfabetizacao_uf', 'meta_alfabetizacao_municipio']:
    print(f"  {nome_tabela}: {'percentual_participacao' in tabelas[nome_tabela].columns}")


Empty DataFrame
Columns: [id_tabela, nome_coluna, chave, cobertura_temporal, valor]
Index: []

percentual_participacao presente nas 3 tabelas de meta:
  meta_alfabetizacao_brasil: True
  meta_alfabetizacao_uf: True
  meta_alfabetizacao_municipio: True


## 16. Validação do Código de Rede para Join com Metas

Confirma, com dado real, qual código de `rede` corresponde à meta pactuada em cada nível (município e UF), e mede a cobertura resultante do join em cada granularidade.

In [16]:
print("=" * 60)
print("  VALIDACAO DO CODIGO DE REDE - NIVEL MUNICIPIO")
print("=" * 60)

df_mun_municipal = tabelas['municipio'][tabelas['municipio']['rede'] == 3]
ids_municipal = set(df_mun_municipal['id_municipio'].unique())
ids_meta_mun = set(tabelas['meta_alfabetizacao_municipio']['id_municipio'].unique())

print(f"\nMunicipios com rede=3 (Municipal): {len(ids_municipal):,}")
print(f"Municipios em meta_alfabetizacao_municipio: {len(ids_meta_mun):,}")
print(f"Sem meta correspondente: {len(ids_municipal - ids_meta_mun):,}")
print(f"Sem resultado correspondente: {len(ids_meta_mun - ids_municipal):,}")

print("\n" + "=" * 60)
print("  VALIDACAO DO CODIGO DE REDE - NIVEL UF")
print("=" * 60)

print(f"\nValores de rede em meta_alfabetizacao_uf: {tabelas['meta_alfabetizacao_uf']['rede'].unique()}")
print(f"\nDistribuicao de rede em uf.csv:")
print(tabelas['uf']['rede'].value_counts())

df_uf_filtrado = tabelas['uf'][tabelas['uf']['rede'] == 5]
ids_uf_filtrado = set(df_uf_filtrado['sigla_uf'].unique())
ids_meta_uf = set(tabelas['meta_alfabetizacao_uf']['sigla_uf'].unique())

print(f"\nUFs com rede=5 (Publica Estadual+Municipal): {len(ids_uf_filtrado)}")
print(f"UFs em meta_alfabetizacao_uf: {len(ids_meta_uf)}")

sem_dado_real = ids_meta_uf - ids_uf_filtrado
sem_meta = ids_uf_filtrado - ids_meta_uf
print(f"UFs com meta mas sem resultado medido (rede=5): {sem_dado_real}")
print(f"UFs com resultado medido mas sem meta: {sem_meta}")

# Confirma que a ausencia e total (nenhum codigo de rede), nao so do codigo 5
print(f"\nLinhas de uf.csv para as UFs sem resultado (qualquer rede):")
print(tabelas['uf'][tabelas['uf']['sigla_uf'].isin(sem_dado_real)][['sigla_uf', 'ano', 'rede']])

  VALIDACAO DO CODIGO DE REDE - NIVEL MUNICIPIO

Municipios com rede=3 (Municipal): 5,500
Municipios em meta_alfabetizacao_municipio: 5,352
Sem meta correspondente: 148
Sem resultado correspondente: 0

  VALIDACAO DO CODIGO DE REDE - NIVEL UF

Valores de rede em meta_alfabetizacao_uf: <ArrowStringArray>
['Pública']
Length: 1, dtype: str

Distribuicao de rede em uf.csv:
rede
3    49
5    49
2    46
0     1
Name: count, dtype: int64

UFs com rede=5 (Publica Estadual+Municipal): 25
UFs em meta_alfabetizacao_uf: 27
UFs com meta mas sem resultado medido (rede=5): {'RR', 'DF'}
UFs com resultado medido mas sem meta: set()

Linhas de uf.csv para as UFs sem resultado (qualquer rede):
Empty DataFrame
Columns: [sigla_uf, ano, rede]
Index: []


## 17. Validaçao da tabela externa com nome de municipios

In [17]:
df_ibge_municipios = tabelas.get('ibge_municipios')

if df_ibge_municipios is not None:
    print("=" * 60)
    print("  VALIDACAO DO CROSSWALK DE NOME DE MUNICIPIO")
    print("=" * 60)

    print(f"\nLinhas: {len(df_ibge_municipios):,}")
    duplicados = df_ibge_municipios['codigo_municipio'].duplicated().sum()
    print(f"Codigos duplicados: {duplicados}")

    tamanhos = df_ibge_municipios['codigo_municipio'].astype(str).str.len().value_counts()
    print(f"\nDistribuicao de digitos do codigo:")
    print(tamanhos)

    ids_crosswalk = set(df_ibge_municipios['codigo_municipio'].astype(str))
    ids_municipio_real = set(tabelas['municipio']['id_municipio'].astype(str))

    print(f"\nCodigos do crosswalk que NAO existem em municipio.csv: {len(ids_crosswalk - ids_municipio_real)}")
    print(f"Codigos de municipio.csv SEM nome no crosswalk: {len(ids_municipio_real - ids_crosswalk)}")

  VALIDACAO DO CROSSWALK DE NOME DE MUNICIPIO

Linhas: 5,571
Codigos duplicados: 0

Distribuicao de digitos do codigo:
codigo_municipio
7    5571
Name: count, dtype: int64

Codigos do crosswalk que NAO existem em municipio.csv: 21
Codigos de municipio.csv SEM nome no crosswalk: 0


## 18. Conclusões e Implicações para o Pipeline

### Achados Principais

| # | Achado | Implicação para o Pipeline |
|---|---|---|
| 1 | `dicionario` é multi-tabela (`id_tabela` + `nome_coluna`) | Join na Silver filtra por ambas antes de cruzar pela `chave` |
| 2 | `municipio` não tem UF | Derivar via `ibge_uf_map.csv` (crosswalk oficial, não hardcode) |
| 3 | `dicionario` e `ibge_uf_map` não têm `ano` | Armazenar na Bronze sem partição temporal |
| 4 | `rede` e `serie` são códigos numéricos em `municipio`/`uf`/`alunos`, mas texto nas tabelas de meta | Decodificar via dicionário antes de qualquer comparação entre as duas origens |
| 5 | `alunos` é a maior tabela em volume (3,8M linhas) | Manter bruta na Bronze, não processar em Silver/Gold |
| 6 | Partição natural por `ano` disponível | Usar `ano` como eixo de partição no data lake |
| 7 | 148 municípios com `rede = Municipal` sem meta correspondente (subconjunto de 5.500) | `LEFT JOIN` (município → meta), flag `possui_meta_municipal`; lacuna real de adesão ao programa, não erro de dado |
| 8 | `rede = 3` (Municipal) é o único código presente em 100% das metas municipais | Filtrar `municipio` por `rede == 3` antes do join com `meta_alfabetizacao_municipio` |
| 9 | ~48% de nulos em `proporcao_aluno_nivel_0..8` em `uf` e `municipio` | Provável sigilo estatístico; manter nulo, não imputar |
| 10 | Nível município cobre apenas 2023-2024; níveis UF/Brasil chegam a 2025 | Gold de evolução temporal no nível município fica limitada a 2 anos reais; não estender artificialmente com metas futuras como se fossem resultado |
| 11 | Nível UF: código `5` (Pública Estadual+Municipal) corresponde à meta, não `3` nem `6` — `6` existe no dicionário mas não ocorre nos dados reais | Filtrar `uf` por `rede == 5` antes do join com `meta_alfabetizacao_uf` |
| 12 | RR e DF têm meta pactuada em `meta_alfabetizacao_uf`, mas nenhuma linha de resultado real em `uf.csv` (nenhum código de rede) | `OUTER JOIN` no nível UF, com flags `possui_resultado_uf` e `possui_meta_uf` — lacuna oposta à de município |
| 13 | **Correção:** nem `Federal` (código 1) nem `Privada` (código 4) ocorrem em `municipio.csv`/`uf.csv`, em nenhum ano. Em `alunos.csv`, `Privada` aparece em apenas 25 de ~3,87M linhas | Datasets de desigualdade por rede não devem incluir Federal/Privada — cobertura real é só Municipal, Estadual e Pública (combinada) |
| 14 | `nivel_alfabetizacao` existe só em `meta_alfabetizacao_municipio`, sem entrada correspondente no dicionário | Mantido como número bruto na Silver/Gold, sem decodificação; escala não documentada pela fonte |
| 15 | `percentual_participacao` presente nas três tabelas de meta (Brasil, UF, Município) | Usar como sinalizador de confiabilidade estatística (ex: `baixa_confiabilidade` quando abaixo de um limiar de referência) na Gold |
